# Lab 8 — Evaluation, Robustness & Shipping 🚀

**Microduck Simulation Course · session 8 of 8**

We own no robot — so **evaluation is our deployment**. Today:
1. Protocols, not vibes: the scorecard
2. The push tournament (a humbling ranking)
3. Sim2sim: variant sweeps as the reality-gap exam
4. The community ecosystem: manifests, reproduction, publishing
5. Mini-projects & where this course goes next

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from microduck_sim.evaluate import evaluate_walking, max_push, variant_sweep
from microduck_sim.policy import OnnxPolicy, PolicyRunner

## Protocols, not vibes

"It looks fine" is not a result. A protocol fixes: the task and command,
the number of episodes and their seeds, the metrics, and the termination
rule — then it's reproducible to the tick (deterministic sim + seeded
resets). Ours (`microduck_sim.evaluate`):

- `evaluate_walking`: survival rate, achieved speed, tracking error over
  seeded perturbed episodes
- `max_push`: largest 0.1 s shove survived while standing
- `variant_sweep`: the same scorecard across *model variants* — our
  substitute for a reality gap

In [ ]:
print("shipped walker @ vx=0.25:", evaluate_walking("walk", episodes=3, duration=6.0))

## The push tournament

Three standing controllers from this course, same exam (numbers from the
committed instructor runs — recompute live if you like):

| controller | origin | max push |
|---|---|---|
| our SB3 `stand_ppo` | Lab 6, 1M steps, 7 min CPU | **0.6 N** |
| hand-tuned ankle PD | Lab 3, ~15 lines | **1.1 N** |
| shipped `stand` policy | GPU, domain-randomized, gets up after falls | **≥ 5.8 N** |

Read it carefully, it's the honest state of robot learning in one table:

- **RL is not magic**: our trained policy loses to a 15-line PD controller.
  Why? Its training env *never contained a push* — you don't get robustness
  you didn't train for.
- **The shipped policy barely has a limit**: it was trained with
  randomization AND it can stand back up — falling stopped being fatal.
  Robustness came from the training *distribution*, not the algorithm.

In [ ]:
print("live check, shipped stand policy:", max_push("stand", resolution=1.0, limit=4.0), "N+")

## Sim2sim: the variant sweep

We can't measure a sim-to-real gap, but we can measure sim-to-*sim*:
the same policy across model variants (full collisions / feet-only /
±1° gear backlash). Recall Lab 7's discovery — on the backlash model the
walker's low-command deadband *disappears* (chatter as dither):

In [ ]:
sweep = variant_sweep("walk", vx=0.25, episodes=2)
for variant, result in sweep.items():
    print(f"{variant:10s} survival {result['survival_rate']:.0%}  "
          f"speed {result['mean_speed']:+.3f}  err {result['tracking_error']:.3f}")

A policy that keeps its score across variants earned its robustness; one
that collapses was overfit to one physics. This is exactly the test the
Isaac-half of the semester will run at scale, with thousands of randomized
worlds.

## The ecosystem: policies as artifacts

The community publishes Microduck behaviors on the Hugging Face Hub as
`policy.onnx` + **`manifest.json`** — obs/action contract, command
semantics, *and a documented eval battery*. Case study,
`RemiFabre/microduck-flamingo-cycle` (stand on one foot on command):

```json
"command": { "twist": ["flag: 0 = two feet, 1 = one foot",
                       "side: +1 = right foot down", "unused"] },
"eval":    { "lifted_foot_height_m": 0.09,
             "known_limits": "falls on backward pushes >= 0.18 m/s ..." }
```

We downloaded it and *reproduced its documented eval in our sim*: commanded
[1, +1] → left foot settles at **0.093 m** (manifest says 0.09), and it
returns to two feet on [0, 0]. Different simulator, same contract, same
behavior — that's what a good manifest buys.

## Publishing your own (the worksheet finale)

Your Lab 6/7 exports are already contract-compatible. Shipping one =
`policy.onnx` + your `manifest.json` (declare commands honestly, document
the eval battery you actually ran, list known limits) + a README, pushed
with `huggingface_hub`:

```python
from huggingface_hub import HfApi
api = HfApi()                       # needs: huggingface-cli login
api.create_repo("YOUR_NAME/microduck-my-behavior", repo_type="model")
api.upload_folder(folder_path="my_release", repo_id="YOUR_NAME/microduck-my-behavior")
```

Known limits are not weakness — the flamingo manifest lists exactly when it
falls, and that's what makes it *usable* by strangers.

## Mini-projects (pick one, 2–3 weeks)

1. **Robust stand**: add random pushes to `MicroduckStand-v0` training;
   beat the PD controller's 1.1 N (target: 2 N+).
2. **Fix the deadband**: retrain walking with commands sampled densely in
   [0.05, 0.25]; deliver a walker with no deadband, manifest documenting
   the tracking curve.
3. **A new trick**: sit-spin, bow, head-bang to a beat — one-shot policy
   with phase encoding (see the crouch-glide's [cos φ, sin φ] convention).
4. **Estimator upgrade**: replace the complementary filter with a Kalman
   filter (Lab 4 bonus), quantify balance improvement under pushes.
5. **Your idea** — pitch it with: task, reward sketch, eval protocol.

Grading = your manifest: does the eval battery hold up when we run it?

## The whole course in one slide

| lab | you built | it became |
|-----|-----------|-----------|
| 1–2 | poses, trajectories, FK | the action space |
| 3 | feedback control | the baseline that RL must beat |
| 4 | estimation from noisy IMU | the observation space |
| 5 | the 61-dim contract | interoperability with the whole ecosystem |
| 6–7 | envs, rewards, PPO | your own policies |
| 8 | protocols & manifests | shippable, checkable artifacts |

Next semester: the same loop — parallel sims, randomization, sim2sim — in
NVIDIA Isaac Sim, at industrial scale. You already speak the language. 🦆